# Análisis de Entidades con GLiNER (Zero-Shot NER)

Este notebook utiliza **GLiNER** para detectar acciones y entidades.

In [ ]:
!pip install gliner

In [ ]:
import json
import pandas as pd
from gliner import GLiNER

print("Cargando modelo GLiNER Large...")
try:
    model = GLiNER.from_pretrained("urchade/gliner_large-v2.1")
    print("Modelo Large cargado exitosamente.")
except Exception as e:
    print(f"No se pudo cargar el modelo Large, intentando con el base: {e}")
    model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
    print("Modelo Base cargado exitosamente.")

In [ ]:
file_path = 'ground_truth_ner.json'

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Se cargaron {len(data)} oraciones para analizar.")

In [17]:
# Definimos un conjunto amplio de etiquetas para capturar la acción
action_labels = [
    "acción", 
    "verbo", 
    "instrucción", 
    "comando",
    "operación",
    "solicitud",
    "tarea"
]

# Etiquetas de contexto
context_labels = [
    "documento",
    "cliente", 
    "producto", 
    "cantidad", 
    "fecha", 
    "identificador"
]

labels = action_labels + context_labels

print(f"Buscando acciones con las etiquetas: {action_labels}")

Buscando acciones con las etiquetas: ['acción', 'verbo', 'instrucción', 'comando', 'operación', 'solicitud', 'tarea']


In [18]:
results = []

for item in data:
    text = item['text']
    
    entities = model.predict_entities(text, labels)
    
    detected_actions = [ent for ent in entities if ent['label'] in action_labels]
    
    formatted_entities = []
    for ent in entities:
        formatted_entities.append({
            'text': ent['text'],
            'label': ent['label'],
            'score': ent['score']
        })
    
    results.append({
        'id': item['id'],
        'text': text,
        'entities': formatted_entities,
        'actions_found': [act['text'] + f" ({act['label']})" for act in detected_actions]
    })

df_results = pd.DataFrame(results)[['id', 'text', 'actions_found']]
pd.set_option('display.max_colwidth', None)
df_results

,id,text,actions_found
0,1,Genera una cotización para el cliente Compufacil con 5 monitores led y 3 soportes de pared,[Genera (comando)]
1,2,Prepara un presupuesto urgente con 10 teclados inalámbricos y 10 ratones ópticos para enviar a Tecnosys,[]
2,3,Crea una oferta comercial para Carla Santana con 1 escritorio ejecutivo modelo XG Premium,[]
3,4,Genera una proforma para AndinaCorp incluye 2 laptops core i7 y 3 impresoras multifunción,[]
4,5,Busca la última factura del cliente Velasco y Asociados y reenvíala a su correo,[]
5,6,Genera la factura del pedido de compra 852025,[pedido de compra (solicitud)]
6,7,Verifica si la factura FA 409516 de Hierros del Pacífico ya está pagada,[Verifica (instrucción)]
7,8,¿Cuántas sillas ergonómicas de oficina tenemos en la bodega de Guayaquil?,[]
8,9,Registra el ingreso de 50 resmas de papel A4 del proveedor Papel Mundo,[]
9,10,Dame el stock actual de discos duros de 1 terabyte de la marca Duradisco,[]


In [ ]:
for item in results:
    print(f"ID: {item['id']}")
    print(f"Texto: {item['text']}")
    
    actions = [ent for ent in item['entities'] if ent['label'] in action_labels]
    others = [ent for ent in item['entities'] if ent['label'] not in action_labels]
    
    if actions:
        print("ACCIONES DETECTADAS:")
        for ent in actions:
            print(f"  >>> {ent['text']} ({ent['label']}) [Confianza: {ent['score']:.4f}]")
    else:
        print("  (No se detectaron acciones)")
        
    if others:
        print("Otras entidades:")
        for ent in others:
            print(f"  - {ent['text']} ({ent['label']})")
            
    print("-" * 50)